# **Portfolio:** Mario Casanova — Data Science & Analytics Portfolio
## **Case Study:** SaaS Product Analytics, Cohorts & Customer Lifetime Value (LTV)

---
*Note: This notebook operationalizes foundational unit economics metrics for business models based on interconnected hardware and subscription services (IoT / SaaS). Using SQL extraction simulations via Pandas, the analysis explores user transactional behavior, deriving conversion rates, cohort drop-offs (Churn Rate), ARPU, and calculating expected Lifetime Value by segmenting by device line.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

import warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
sns.set_palette("crest")

### 1. Simulated Data Warehouse Ingestion (Hardware & Subscriptions)
We load the transactional database of device sales (Security Cameras vs Smart Doorbells) and cloud storage contractual records (Premium Subscriptions).

In [ ]:
# Dimension Table: Users and Installed Base Inventory
df_users = pd.read_csv('../data/hardware_users.csv')
df_users['PurchaseDate'] = pd.to_datetime(df_users['PurchaseDate'])

# Fact Table: Subscription Records
df_subs = pd.read_csv('../data/subscriptions.csv')
df_subs['SubscriptionStart'] = pd.to_datetime(df_subs['SubscriptionStart'])
df_subs['SubscriptionEnd'] = pd.to_datetime(df_subs['SubscriptionEnd'])

# Hybrid Join: Enrich installed base with subscription status
# SQL Equivalent: SELECT * FROM Users LEFT JOIN Subs ON Users.UserID = Subs.UserID
df_analytics = pd.merge(df_users, df_subs, on='UserID', how='left')

print(f"[*] Total Installed Base: {len(df_users)} Active Hardware.")
print(f"[*] Subscriber Base: {len(df_subs)} Processed contracts.")
df_analytics.head()

### 2. Conversions and Hardware-to-Subscription Attach Rate
The *Attach Rate* measures the effectiveness of the *in-app* funnel in converting a recent physical buyer into a recurring value user.

In [ ]:
total_hardware = len(df_users)
total_subscriptions = len(df_subs)
global_attach_rate = (total_subscriptions / total_hardware) * 100

print(f"[KPI] Global Hardware-to-Subscription Attach Rate: {global_attach_rate:.2f}%")

conv_by_device = df_analytics.groupby('DeviceType').agg(
    Total_Hardware=('UserID', 'count'),
    Subscribers=('SubscriptionStart', 'count')
)
conv_by_device['Attach_Rate_Pct'] = (conv_by_device['Subscribers'] / conv_by_device['Total_Hardware']) * 100
conv_by_device


### 3. Annualized Customer Churn Rate by Segment
We probabilistically define the retention rate. Operationally: Lost Subscribers / Unique Total Subscribers within the evaluated period.

In [ ]:
# We only evaluate users who actually signed up
df_subscribers = df_analytics[df_analytics['SubscriptionStart'].notna()].copy()

churn_analysis = df_subscribers.groupby('DeviceType').agg(
    Starts=('UserID', 'count'),
    Cancellations=('IsActive', lambda x: (x == False).sum())
)

churn_analysis['Churn_Rate_Pct'] = (churn_analysis['Cancellations'] / churn_analysis['Starts']) * 100

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=churn_analysis.reset_index(), x='DeviceType', y='Churn_Rate_Pct', palette=['#ff5252', '#4db6ac'], ax=ax)
ax.set_title("Customer Churn Rate by Hardware Line", fontsize=14, pad=10)
ax.set_ylabel("Cancellation Index (% Churn)")
ax.set_xlabel("")
for i, v in enumerate(churn_analysis['Churn_Rate_Pct']):
    ax.text(i, v - 3, f"{v:.1f}%", ha='center', color='white', fontweight='bold', fontsize=12)
plt.show()


### 4. LTV (Lifetime Value) Projection and Segmented ARPU
A Fixed Contribution Margin for Cloud Storage is assumed (e.g., 80%). 
The governing formula: $ LTV = \frac{ARPU \times Margin}{Churn \; Rate} $

In [ ]:
GROSS_MARGIN = 0.80 # 80% gross margin on software/cloud

# ARPU (Average Revenue Per User monthly) assuming linear billings
arpu_data = df_subscribers.groupby('DeviceType').agg(ARPU=('MonthlyFee', 'mean'))

# Unit Economic Consolidation
economics = pd.concat([arpu_data, churn_analysis[['Churn_Rate_Pct']]], axis=1)
economics['Churn_Decimal'] = economics['Churn_Rate_Pct'] / 100

# LTV Projection
economics['LTV_Projected_USD'] = (economics['ARPU'] * GROSS_MARGIN) / economics['Churn_Decimal']

economics[['ARPU', 'Churn_Rate_Pct', 'LTV_Projected_USD']].round(2)

### 5. Cohort Analysis (Structural Heatmap)
Cohort analysis allows us to audit whether product changes, marketing, or onboarding flows launched in specific months impacted Attach Rate absorption over time.

In [ ]:
cohort_data = df_analytics.groupby(['CohortMonth', 'DeviceType']).agg(
    Total_Hardware=('UserID', 'count'),
    Subscribers=('SubscriptionStart', 'count')
).reset_index()

cohort_data['Attach_Rate'] = (cohort_data['Subscribers'] / cohort_data['Total_Hardware'])

cohort_pivot = cohort_data.pivot(index='CohortMonth', columns='DeviceType', values='Attach_Rate')

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cohort_pivot, annot=True, fmt=".1%", cmap="YlGnBu_r", linewidths=.5, cbar_kws={'label': 'SaaS Conversion Rate'})
ax.set_title("Cohort Evolution of Subscription Attach Rate (2022)", fontsize=15, pad=15)
ax.set_ylabel("Acquisition Cohort (Purchase Month)")
ax.set_xlabel("Hardware Core")
plt.show()

### Synthesis for Budget Allocation (CAC)
The data overwhelmingly shows that while *Security Cameras* generate moderately higher monthly fees, their **low propensity to cancel (Churn Rate)** gives them a Lifetime Value ($LTV$) almost three times higher than that of Smart Doorbells. 

For the Marketing / Paid Media team, this empirically means that **we can tolerate a much more aggressive Customer Acquisition Cost (CAC)** in camera campaigns than in the doorbell line, maintaining a healthy LTV:CAC ratio (> 3:1).